In [ ]:
# Alap importok
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import time #új
import random #új
simulator = AerSimulator()

In [ ]:
n = 3
secret_string = '101'   # Rejtett bitvektor (s)

qc = QuantumCircuit(n+1, n)   # n bemeneti qubit + 1 segédqubit + n klasszikus bit

# Inicializálás: minden bemeneti qubit |+⟩ állapotba (Hadamard)
for i in range(n):
    qc.h(i)

# Segédqubit |−⟩ állapotba (X majd H)
qc.x(n)
qc.h(n)

qc.barrier() # Csak vizuális elválasztás – nem befolyásolja az eredményt

In [ ]:
# Orákulum implementációja
for i in range(n):
    if secret_string[i] == '1':
        qc.cx(i, n)   # ha s_i = 1, akkor CNOT az i-edik qubitről a segédqubitre

qc.barrier()

In [ ]:
# Hadamard minden bemeneti qubiten (interferencia)
for i in range(n):
    qc.h(i)

# Mérjük a bemeneti qubiteket
qc.measure(range(n), range(n))

display(qc.draw('mpl'))

In [ ]:
job = simulator.run(qc, shots=1024)
result = job.result()
counts = result.get_counts()

print("Mérési eredmények:", counts)
plot_histogram(counts)

In [ ]:
# ---Skálázhatóság vizsgálata--- (Új)

n_values = [3, 5, 8, 12]

print(f"{'n':<5} | {'Klasszikus Q':<12} | {'Kvantum Q':<10} | {'Szimulációs idő (s)':<15}")
print("-" * 55)

for test_n in n_values:
    test_s = '1' * test_n
    test_qc = QuantumCircuit(test_n+1, test_n)
    test_qc.h(range(test_n))
    test_qc.x(test_n)
    test_qc.h(test_n)
    for i in range(test_n):
        if test_s[i] == '1': test_qc.cx(i, test_n)
    test_qc.h(range(test_n))
    test_qc.measure(range(test_n), range(test_n))
    
    start = time.time()
    simulator.run(transpile(test_qc, simulator), shots=1).result()
    end = time.time()
    
    print(f"{test_n:<5} | {test_n:<12} | {1:<10} | {end-start:.6f}")

plt.figure(figsize=(8, 4))
plt.plot(n_values, n_values, label='Klasszikus lekérdezés (n)', marker='o', linestyle='--')
plt.plot(n_values, [1]*len(n_values), label='Kvantum lekérdezés (1)', marker='s', linewidth=2)
plt.xlabel("n (titkos string hossza)")
plt.ylabel("Lekérdezések száma")
plt.title("Bernstein–Vazirani skálázhatóság: Kvantum előny")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

# ---Részleges orákulum--- (Új)

n_partial = 5
secret_partial = '11111'
print(f"Eredeti titkos string: {secret_partial}")

qc_p = QuantumCircuit(n_partial+1, n_partial)
qc_p.h(range(n_partial))
qc_p.x(n_partial)
qc_p.h(n_partial)
qc_p.barrier()

error_index = random.randint(0, n_partial-1)
for i in range(n_partial):
    if i == error_index:
        print(f"Hiba történt a(z) {i}. indexnél: CNOT kihagyva/megfordítva.")
        continue 
    if secret_partial[i] == '1':
        qc_p.cx(i, n_partial)

qc_p.barrier()
qc_p.h(range(n_partial))
qc_p.measure(range(n_partial), range(n_partial))

counts_p = simulator.run(transpile(qc_p, simulator), shots=1024).result().get_counts()
print(f"Mért eredmény: {list(counts_p.keys())[0]}")
plot_histogram(counts_p)

In [ ]:
# ---Klasszikus vs. kvantum lekérdezésszám kísérlet--- (Új)

def classical_bv_solver(secret_string):
    n = len(secret_string)
    queries = 0
    found_secret = ""
    
    for i in range(n):
        queries += 1
        found_secret += secret_string[i] 
        
    return queries

n_range = range(1, 16)
c_results = []
q_results = []

for n in n_range:
    s = '1' * n
    c_results.append(classical_bv_solver(s))
    q_results.append(1)

print(f"{'n (bitek)':<10} | {'Klasszikus lépés':<18} | {'Kvantum lépés':<15}")
print("-" * 50)
for i in range(len(n_range)):
    if n_range[i] in [1, 5, 10, 15]:
        print(f"{n_range[i]:<10} | {c_results[i]:<18} | {q_results[i]:<15}")